# 05. v2 — 차트 Vision 캡션

v1까지는 차트 숫자가 `14.5 15.0 11.5 12.7`처럼 **어느 연도의 값인지 모르는 숫자 나열**로만 들어갔습니다.
v2는 차트 영역을 이미지로 잘라 **Vision LLM**에게 보여주고, `- 국내관광 · 2015년 = 66.7%` 같은 **캡션(설명 글)**을 받아 검색 대상에 추가합니다.

- 문서 읽기 코드: `app/ingest/parse.py` (v1 노트북의 과정을 함수로 옮김), 차트 코드: `app/ingest/charts.py`
- **v1과 같게 두는 것:** 답변 프롬프트, 답변 모델, TOP_K=5, 청크 크기, 채점기, 테스트셋
- **API 사용 (요금 발생)** 칸: ③ 캡션 생성(약 $0.2), ⑤ 인덱스(약 $0.006), ⑥ 평가(약 $0.09)
- 캡션·인덱스·평가 결과는 저장해 두고, 다시 실행하면 **불러오기만** 합니다.

## ① 설정

In [1]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", message=".*langchain-community.*")

import pandas as pd
from langchain_core.callbacks import get_usage_metadata_callback
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))

from app.config import settings
from app.ingest.parse import parse_pdf, caption_charts, build_chunks
from eval.rag_eval import load_testset, run_eval, summarize, JudgeResult

TOP_K = 5
INDEX_DIR = settings.data_dir / "index" / "v2_faiss"
RESULTS_DIR = PROJECT_DIR / "eval" / "results"

DOCS = {
    "D01": "2026_youth_employment_survey.pdf",
    "D02": "2025_content_industry_trend.pdf",
    "D03": "2025_social_survey.pdf",
}
PAGE_RANGES = {"D01": [(1, 40)], "D02": [(13, 22), (37, 54)], "D03": [(1, 56)]}

print("API 키:", "있음" if settings.openai_api_key else "없음 (.env 확인)")
print("답변:", settings.llm_model, "| Vision:", settings.vision_model, "| 채점:", settings.judge_model,
      "| 임베딩:", settings.embedding_model)

API 키: 있음
답변: gpt-6-luna | Vision: gpt-6.1-terra | 채점: gpt-6.1-terra | 임베딩: text-embedding-3-small


## ② 문서 읽기 + 차트 영역 찾기
v1과 같은 방법으로 본문·표를 읽고, 추가로 **차트 영역**을 찾습니다.
- 그림 차트: PDF 안의 이미지 (D01은 가는 띠 수백 개로 잘려 있어 붙어 있는 것끼리 합침)
- 벡터 차트: 표 테두리를 뺀 도형(선·막대)이 모여 있는 곳 (D03)
- 찾은 영역은 페이지 폭 전체의 **띠**로 넓혀 자릅니다. 나란히 놓인 차트 두 개와 차트 제목을 함께 담기 위해서입니다.
- 표로 찾았지만 행 이름이 대부분 비어 있는 것(차트를 표로 잘못 찾은 것)은 표에서 빼고 차트로 보냅니다.

In [2]:
parsed = {doc_id: parse_pdf(settings.pdf_dir / f, doc_id, PAGE_RANGES[doc_id]) for doc_id, f in DOCS.items()}
for doc_id, p in parsed.items():
    pages = sorted({j.page for j in p.chart_jobs})
    print(f"{doc_id}: 표 {p.n_tables}개 | 차트 띠 {len(p.chart_jobs)}개 (쪽: {pages})")

D01: 표 56개 | 차트 띠 9개 (쪽: [5, 6, 7, 10, 12, 20, 25, 30])
D02: 표 13개 | 차트 띠 5개 (쪽: [44, 46, 48, 50, 52])
D03: 표 55개 | 차트 띠 33개 (쪽: [4, 5, 8, 9, 10, 11, 12, 14, 16, 17, 21, 22, 23, 24, 25, 28, 32, 33, 34, 36, 38, 39, 40, 42, 43, 46, 47, 48, 49, 50, 51, 53, 55])


## ③ 차트 캡션 생성 — API 사용 (요금 발생, 약 $0.2)
차트 이미지는 `data/assets/`, 캡션은 `data/captions/`에 저장합니다. 저장된 캡션이 있으면 Vision을 다시 부르지 않습니다.

In [3]:
vision_llm = ChatOpenAI(model=settings.vision_model)
with get_usage_metadata_callback() as usage:
    for doc_id, p in parsed.items():
        n = caption_charts(p, vision_llm, settings.caption_dir, settings.asset_dir, settings.vision_model)
        print(f"{doc_id}: 새로 만든 캡션 {n}개 / 전체 {len(p.chart_jobs)}개")
for model, u in usage.usage_metadata.items():
    cost = u["input_tokens"] / 1e6 * 1 + u["output_tokens"] / 1e6 * 5   # gpt-6.1-terra: 입력 $1, 출력 $5 (100만 토큰당)
    print(f"{model}: 입력 {u['input_tokens']:,} / 출력 {u['output_tokens']:,} 토큰 (약 ${cost:.3f})")

D01: 새로 만든 캡션 9개 / 전체 9개


D02: 새로 만든 캡션 5개 / 전체 5개


D03: 새로 만든 캡션 33개 / 전체 33개
gpt-6.1-terra: 입력 39,450 / 출력 27,920 토큰 (약 $0.179)


## ④ 캡션 확인
v1에서 틀렸던 차트 문항의 쪽을 봅니다. (D01 10쪽 연령별 인구 증감, D03 16쪽 사회보험료 부담)

In [4]:
def show_caption(doc_id, page_no, n_lines=12):
    for job in parsed[doc_id].chart_jobs:
        if job.page == page_no:
            print(f"===== {doc_id} p.{page_no}  이미지: {Path(job.image_path).name}")
            print("\n".join(job.caption.splitlines()[:n_lines]))
            print()

show_caption("D01", 10)
show_caption("D03", 16)

===== D01 p.10  이미지: p010_03.png
[차트] 연령별 인구 증감 추이
- 15-19세 · '22.5 = -74천명
- 20-24세 · '22.5 = -93천명
- 25-29세 · '22.5 = -37천명
- 15-19세 · '23.5 = 18천명
- 20-24세 · '23.5 = -110천명
- 25-29세 · '23.5 = -87천명
- 15-19세 · '24.5 = -31천명
- 20-24세 · '24.5 = -145천명
- 25-29세 · '24.5 = -67천명
- 15-19세 · '25.5 = 2천명
- 20-24세 · '25.5 = -140천명

===== D01 p.10  이미지: p010_04.png
[차트] 최종학교 졸업자 연령별 증감 추이
- 15-19세 · '22.5 = 3천명
- 20-24세 · '22.5 = -8천명
- 25-29세 · '22.5 = -10천명
- 15-19세 · '23.5 = -22천명
- 20-24세 · '23.5 = -72천명
- 25-29세 · '23.5 = -76천명
- 15-19세 · '24.5 = 42천명
- 20-24세 · '24.5 = -113천명
- 25-29세 · '24.5 = -84천명
- 15-19세 · '25.5 = -24천명
- 20-24세 · '25.5 = -118천명

===== D03 p.16  이미지: p016_08.png
[차트] 사회보험료 부담정도
- 건강보험 · 2015년 = 64.5%
- 건강보험 · 2017년 = 61.7%
- 건강보험 · 2019년 = 56.0%
- 건강보험 · 2023년 = 57.2%
- 건강보험 · 2025년 = 55.3%
- 국민연금 · 2015년 = 63.4%
- 국민연금 · 2017년 = 59.3%
- 국민연금 · 2019년 = 55.8%
- 국민연금 · 2023년 = 57.1%
- 국민연금 · 2025년 = 58.4%
- 고용보험 · 2015년 = 39.3%



## ⑤ 청크 + 인덱스 — API 사용 (약 $0.006)
본문·표 청크(v1과 같은 방식)에 차트 캡션 청크를 더합니다. 차트 청크는 `[차트] [섹션] 차트 제목` 머리말 + 값 줄들입니다.

In [5]:
chunks = []
for p in parsed.values():
    chunks += build_chunks(p, settings.chunk_size, settings.chunk_overlap)
kinds = pd.Series([c.metadata["kind"] for c in chunks]).value_counts()
print("청크 수:", len(chunks), kinds.to_dict())
print("\n[차트 청크 예시]\n" + next(c.page_content for c in chunks if c.metadata["kind"] == "chart")[:400])

embeddings = OpenAIEmbeddings(model=settings.embedding_model)
if INDEX_DIR.exists():
    vectorstore = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)  # 직접 만든 파일
    print("\n불러옴:", INDEX_DIR)
else:
    vectorstore = FAISS.from_documents(chunks, embeddings)
    vectorstore.save_local(str(INDEX_DIR))
    print("\n만들고 저장:", INDEX_DIR)
print("인덱스 청크 수:", vectorstore.index.ntotal)

청크 수: 610 {'table': 357, 'text': 152, 'chart': 101}

[차트 청크 예시]
[차트] [Ⅱ. 2026년 5월 청년층(15~29세) 부가조사 결과 > 2. 대학 졸업 소요기간 및 휴학상태] 청년층(15~29세)
- 인구 · 15세 이상 = 45,984천명
- 인구 · 청년층 = 7,822천명
- 인구 비율 · 청년층 = 17.0%



만들고 저장: /home/user/git_project/project6/ReportQA/data/index/v2_faiss
인덱스 청크 수: 610


## ⑥ 평가 — API 사용 (약 $0.09)
v0·v1과 **같은 프롬프트, 같은 채점기**로 110문항을 평가해 `eval/results/v2.csv`에 저장합니다.

In [6]:
llm = ChatOpenAI(model=settings.llm_model)

# 02~04 노트북과 같은 프롬프트 (버전 간 비교를 위해 바꾸지 않음)
PROMPT = """당신은 통계 보고서 내용으로만 답하는 도우미입니다.
규칙:
1. 아래 [근거]에 있는 내용으로만 답하세요. 추측하지 마세요.
2. 답의 끝에 사용한 근거를 [문서ID p.쪽] 형식으로 표시하세요. 예: [D01 p.8]
3. [근거]에서 답을 찾을 수 없으면 정확히 "모름"이라고만 답하세요.

[근거]
{context}

[질문]
{question}"""

def answer_v2(question):
    found = vectorstore.similarity_search(question, k=TOP_K)
    context = "\n\n".join(f"[{d.metadata['doc_id']} p.{d.metadata['page']}]\n{d.page_content}" for d in found)
    response = llm.invoke(PROMPT.format(context=context, question=question))
    return response.text, [(d.metadata["doc_id"], d.metadata["page"]) for d in found]

testset = load_testset(PROJECT_DIR / "eval" / "testset.csv")
RESULT_PATH = RESULTS_DIR / "v2.csv"
if RESULT_PATH.exists():
    results = pd.read_csv(RESULT_PATH)
    print("불러옴:", RESULT_PATH)
else:
    judge_llm = ChatOpenAI(model=settings.judge_model).with_structured_output(JudgeResult)
    with get_usage_metadata_callback() as usage:
        results = run_eval(testset, answer_v2, judge_llm, k=TOP_K)
    results.to_csv(RESULT_PATH, index=False)
    print("저장:", RESULT_PATH)
    for model, u in usage.usage_metadata.items():
        print(f"{model}: 입력 {u['input_tokens']:,} / 출력 {u['output_tokens']:,} 토큰")

10/110 완료


20/110 완료


30/110 완료


40/110 완료


50/110 완료


60/110 완료


70/110 완료


80/110 완료


90/110 완료


100/110 완료


110/110 완료
저장: /home/user/git_project/project6/ReportQA/eval/results/v2.csv
gpt-6-luna: 입력 223,696 / 출력 4,725 토큰
gpt-6.1-terra: 입력 29,695 / 출력 3,400 토큰


## ⑦ v0 · v1 · v2 비교

In [7]:
versions = {v: pd.read_csv(RESULTS_DIR / f"{v}.csv") for v in ["v0", "v1"]}
versions["v2"] = results
compare = pd.DataFrame({v: summarize(r) for v, r in versions.items()})
compare["v1→v2(%p)"] = ((compare["v2"] - compare["v1"]) * 100).round(1)
print("판정 분포 v2:", results["judge"].value_counts().to_dict())
compare

판정 분포 v2: {'correct': 97, 'refused': 10, 'wrong': 3}


,v0,v1,v2,v1→v2(%p)
Recall@5,0.870,0.890,0.920,3.0
정답률,0.673,0.736,0.882,14.6
텍스트,0.925,0.950,0.950,0.0
표,0.533,0.833,0.900,6.7
차트(vector),0.611,0.444,0.833,38.9
차트(image),0.000,0.000,0.583,58.3
환각률,0.000,0.000,0.000,0.0


In [8]:
pd.set_option("display.max_colwidth", 70)
both = (versions["v1"][["id", "type", "chart_kind", "judge"]]
        .merge(results[["id", "judge", "recall_hit", "answer_text"]], on="id", suffixes=("_v1", "_v2"))
        .merge(testset[["id", "question", "answer"]], on="id"))
improved = both[(both["judge_v1"] != "correct") & (both["judge_v2"] == "correct")]
regressed = both[(both["judge_v1"] == "correct") & (both["judge_v2"] != "correct")]
print(f"좋아진 문항 {len(improved)}개 / 나빠진 문항 {len(regressed)}개")
display(improved[["id", "type", "chart_kind", "question"]])
display(regressed[["id", "type", "question", "answer", "answer_text", "recall_hit"]])

좋아진 문항 17개 / 나빠진 문항 1개


,id,type,chart_kind,question
14,B002,table,NaN,2026년 5월 25~29세 경제활동참가율은?
17,B005,table,NaN,2026년 5월 3년제 이하 여자 대졸자의 평균 졸업 소요기간은?
22,B010,table,NaN,2026년 5월 취업시험 준비 분야 중 교원임용 비율은?
26,C004,chart,image,2024년 5월 25~29세 인구의 전년동월대비 증감은?
28,C006,chart,image,2022~2026년(매년 5월) 중 20~24세 인구 감소폭이 가장 컸던 해는?
29,C007,chart,image,2023년 5월 25~29세 최종학교 졸업자의 전년동월대비 증감은?
30,C008,chart,image,2024년 5월 15~19세 최종학교 졸업자의 전년동월대비 증감은?
32,C010,chart,image,2025년 5월 25~29세 최종학교 졸업자의 전년동월대비 증감은?
33,C011,chart,image,2022년 5월 청년층 비경제활동인구 중 취업시험 준비자 비율은?
34,C012,chart,image,2024년 5월 청년층 비경제활동인구 중 취업시험 준비자 비율은?


,id,type,question,answer,answer_text,recall_hit
13,B001,table,2026년 5월 15세 이상 전체 실업률은?,2.9%,모름,False


In [9]:
# v2에서도 틀린 차트 문항
still = both[(both["type"] == "chart") & (both["judge_v2"] != "correct")]
print("v2에서도 틀린 차트 문항:", len(still), "개")
still[["id", "chart_kind", "question", "answer", "answer_text", "recall_hit"]]

v2에서도 틀린 차트 문항: 8 개


,id,chart_kind,question,answer,answer_text,recall_hit
23,C001,image,2022년 5월 15~19세 인구의 전년동월대비 증감은?,-74천명,모름,False
24,C002,image,2023년 5월 15~19세 인구의 전년동월대비 증감은?,+18천명,모름,False
25,C003,image,2023년 5월 20~24세 인구의 전년동월대비 증감은?,-110천명,모름,False
27,C005,image,2025년 5월 15~19세 인구의 전년동월대비 증감은?,+2천명,모름,False
31,C009,image,2025년 5월 20~24세 최종학교 졸업자의 전년동월대비 증감은?,-118천명,모름,False
86,C017,vector,2019년 60세 이상 고령자 중 현재 자녀와 따로 사는 비중은?,70.7%,68.4%입니다. [D03 p.22],True
90,C021,vector,2019년 자신의 사회·경제적 지위를 '중'이라고 생각하는 비율은?,58.5%,모름,True
95,C026,vector,2015년 가구 소득수준이 부족하다고 응답한 가구주 비율은?,64.7%,2015년 가구 소득수준이 부족하다고 응답한 가구주 비율은 27.0%입니다. [D03 p.46],True


## ⑧ 발견한 것

### 1. 점수 변화
| 지표 | v0 | v1 | **v2** | v1→v2 |
|---|---|---|---|---|
| 정답률 | 67.3% | 73.6% | **88.2%** | +14.6%p |
| 텍스트 | 92.5% | 95.0% | 95.0% | 0 |
| 표 | 53.3% | 83.3% | **90.0%** | +6.7%p |
| 차트(벡터) | 61.1% | 44.4% | **83.3%** | **+38.9%p** |
| 차트(이미지) | 0% | 0% | **58.3%** | **+58.3%p** |
| Recall@5 | 87.0% | 89.0% | 92.0% | +3.0%p |
| 환각률 | 0% | 0% | 0% | 0 |

**계획서의 "증명 목표"를 달성했습니다.** 텍스트만 쓰는 RAG(v0)에서 0%였던 그림 차트 문항이 Vision 캡션으로 58.3%가 됐고,
v1에서 연도 순서를 몰라 흔들리던 벡터 차트 문항도 83.3%로 안정됐습니다. 좋아진 17문항 중 14문항이 차트, 나빠진 문항은 1개(B001)입니다.

### 2. 왜 좋아졌나 — "숫자 나열"이 "계열 · 연도 = 값"으로
```
v1 (본문 추출): 70%  60.8  64.5  60%  59.4  61.7  57.5 …  2015  2017  2019  2023  2025
v2 (Vision 캡션): - 건강보험 · 2015년 = 64.5%   - 건강보험 · 2017년 = 61.7% …
```
- 그림 차트(D01)는 PDF 안에 글자가 아예 없어서 v0·v1에서는 검색 대상조차 아니었습니다.
- 표에서도 3문항이 좋아졌습니다(B002·B005·B010). 같은 숫자가 차트에도 그려져 있어, 캡션이 **두 번째 근거**가 됐습니다.

### 3. 캡션 품질
- 차트 띠 47개, 캡션 비용 **약 $0.18** (입력 39,450 / 출력 27,920 토큰). 문서당 1번만 만들고 `data/captions/`에 저장해 재사용합니다.
- 캡션을 직접 확인한 결과(D01 10쪽, D03 16·22·38·46쪽) 데이터 레이블은 대부분 정확히 옮겨졌습니다.
- **오독 사례 1건:** D03 16쪽 꺾은선 차트에서 2019년 건강보험(55.8%)과 국민연금(56.0%) 값이 **서로 바뀌어** 기록됨. 두 선의 레이블이 가까이 겹쳐 있는 곳입니다. (계획서 13장 "Vision 오독" 위험이 실제로 나타난 사례 → 근거 화면에서 원본 이미지를 함께 보여주는 이유)

### 4. 남은 차트 오답 8문항
| 원인 | 문항 | 설명 |
|---|---|---|
| **검색 실패** | C001·C002·C003·C005·C009 (그림) | 캡션은 `'22.5` (22년 5월) 형식인데 질문은 "2022년 5월" → 단어도 뜻도 잘 안 맞아 상위 5개에 못 듦 → **v3 BM25·리랭커 대상** |
| 찾았지만 오답 | C017·C026 | 캡션 값은 정확(70.7%, 64.7%)하지만 다른 연도·계열 값을 골라 답함 |
| 찾았지만 "모름" | C021 | 캡션에 `중 · 2019년 = 58.5%`가 있으나 "모름" |

### 5. 다음 단계 (v3)
검색 실패가 남은 오답의 대부분입니다. → 6주차: BM25 키워드 검색 + 리랭커로 "검색 순위"를 개선합니다.